# Reproduce Best Climate-Risk Submission

This notebook reproduces the best confirmed submission using the available training, test, and climate feature files. It fits two balanced Logistic Regression models, blends their probabilities, and writes `best_submission_reproduced.csv`.

Model blend: 70% with `C=2.0` and 30% with `C=1.0`. `TargetF1` cutoff: `0.275`.

In [4]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_predict

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
if not (ROOT / "Train.csv").exists():
    ROOT = ROOT / "Primary competition visual Climate Risk and Health Prediction Challenge"
TARGET, ID = "is_climate_sensitive", "ID"

train = pd.read_csv(ROOT / "Train.csv")
test = pd.read_csv(ROOT / "Test.csv")
if (ROOT / "climate_features.csv").exists():
    c = pd.read_csv(ROOT / "climate_features.csv").drop(columns=["deathdate"], errors="ignore")
    c = c.drop_duplicates(subset=[ID], keep="last")
    train, test = train.merge(c, on=ID, how="left"), test.merge(c, on=ID, how="left")

print("=== SHAPES / TARGET ===")
print(train.shape, test.shape, "positive rate:", train[TARGET].mean().round(4))
print("\n=== COLUMNS (dtype, %null train, %null test) ===")
for col in train.columns:
    if col in test.columns:
        print(f"{col:30s} {str(train[col].dtype):10s} {train[col].isna().mean():.2f} {test[col].isna().mean():.2f}")

print("\n=== CATEGORY OVERLAP (share of test rows whose value exists in train) ===")
for col in train.select_dtypes(exclude="number").columns:
    if col in test.columns and col != ID:
        seen = test[col].isin(set(train[col].dropna())).mean()
        print(f"{col:30s} nunique train={train[col].nunique():6d} test-seen={seen:.3f}")

# Encode everything numerically for quick GBDT checks
def encode(df):
    d = df.drop(columns=[ID, TARGET], errors="ignore").copy()
    if "deathdate" in d.columns:
        dt = pd.to_datetime(d["deathdate"], errors="coerce")
        d["death_year"], d["death_month"] = dt.dt.year, dt.dt.month
        d["death_dow"], d["death_doy"] = dt.dt.dayofweek, dt.dt.dayofyear
        d = d.drop(columns=["deathdate"])
    for col in d.columns:
        if not pd.api.types.is_numeric_dtype(d[col]) and not pd.api.types.is_bool_dtype(d[col]):
            d[col] = d[col].astype(str).astype("category").cat.codes
    return d.astype(float)

allx = encode(pd.concat([train, test], ignore_index=True))
Xtr, Xte = allx.iloc[:len(train)], allx.iloc[len(train):]
y = train[TARGET].astype(int).values

print("\n=== ADVERSARIAL VALIDATION (train vs test; 0.5 = identical, >0.7 = big shift) ===")
adv_X = pd.concat([Xtr, Xte]); adv_y = np.r_[np.zeros(len(Xtr)), np.ones(len(Xte))]
m = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, verbose=-1)
p = cross_val_predict(m, adv_X, adv_y, cv=StratifiedKFold(5, shuffle=True, random_state=0), method="predict_proba")[:, 1]
print("Adversarial AUC:", round(roc_auc_score(adv_y, p), 4))
m.fit(adv_X, adv_y)
imp = pd.Series(m.feature_importances_, adv_X.columns).sort_values(ascending=False)
print("Top shifting features:\n", imp.head(8))

print("\n=== CV: RANDOM vs GROUPED (leakage check) ===")
def cv_auc(X, cv, groups=None):
    mm = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03, verbose=-1)
    pr = cross_val_predict(mm, X, y, cv=cv, groups=groups, method="predict_proba")[:, 1]
    return round(roc_auc_score(y, pr), 4)

print("Random 5-fold, all features:", cv_auc(Xtr, StratifiedKFold(5, shuffle=True, random_state=0)))
no_loc = Xtr.drop(columns=[c for c in ["location"] if c in Xtr.columns])
print("Random 5-fold, no location :", cv_auc(no_loc, StratifiedKFold(5, shuffle=True, random_state=0)))
for g in ["location", "zone"]:
    if g in train.columns:
        grp = train[g].fillna("na").astype(str).values
        grp = train[g].fillna("na").astype(str).values
        n_groups = pd.unique(grp).size
        if n_groups >= 2:
            n_splits = min(5, n_groups)
            print(f"GroupKFold by {g:9s}, no location:", cv_auc(no_loc, GroupKFold(n_splits=n_splits), grp))
        else:
            print(f"GroupKFold by {g:9s}, skipped (only {n_groups} group(s))")

=== SHAPES / TARGET ===
(3146, 29) (1030, 28) positive rate: 0.6507

=== COLUMNS (dtype, %null train, %null test) ===
ID                             str        0.00 0.00
zone                           str        0.00 0.00
gender                         str        0.00 0.00
deathdate                      str        0.00 0.00
age                            float64    0.00 0.00
avg_temperature                float64    0.00 0.00
max_temperature                float64    0.00 0.00
min_temperature                float64    0.00 0.00
precipitation                  float64    0.00 0.00
latitude                       float64    0.00 0.00
longitude                      float64    0.00 0.00
location                       str        0.00 0.00
elevation                      int64      0.00 0.00
hot_days_30d                   int64      0.00 0.00
max_daily_rain_30d             float64    0.00 0.00
ndvi_30d                       float64    0.00 0.00
ndvi_90d                       float64    0.00 0.0